## Consumo y Carga de Datos desde API REST con Python y Pandas

Una **API REST (Representational State Transfer)** es una interfaz estándar que permite la comunicación e intercambio de datos estructurados entre aplicaciones a través del protocolo HTTP/HTTPS.

### Conceptos Clave:
- **Endpoint**: Dirección URL donde reside un recurso específico (ej. `/posts`, `/users`).
- **Métodos HTTP**: `GET` (consultar), `POST` (crear), `PUT`/`PATCH` (modificar), `DELETE` (eliminar).
- **Códigos de Estado HTTP**:
  - `200 OK`: Petición exitosa.
  - `201 Created`: Recurso creado satisfactoriamente.
  - `400 Bad Request`: Solicitud errónea del cliente.
  - `404 Not Found`: Recurso no encontrado.
  - `500 Internal Server Error`: Fallo en el servidor.
- **Formato JSON**: Formato de texto ligero basado en pares clave-valor.

En esta práctica utilizaremos **{JSON} Placeholder** (`https://jsonplaceholder.typicode.com`), una API pública y gratuita diseñada para pruebas y desarrollo.

In [ ]:
%pip install requests pandas

In [ ]:
import requests
import pandas as pd
import json

# URL base de la API REST de pruebas JSONPlaceholder
BASE_URL = "https://jsonplaceholder.typicode.com"
print(f"API Base URL: {BASE_URL}")

## 1. Petición GET al Endpoint de Publicaciones (`/posts`)

In [ ]:
# Realizar petición HTTP GET para obtener las publicaciones
url_posts = f"{BASE_URL}/posts"
response = requests.get(url_posts, timeout=10)

print(f"URL consultada: {response.url}")
print(f"Código de estado HTTP: {response.status_code}")
print(f"Encabezado Content-Type: {response.headers.get('Content-Type')}")

# Verificar si la respuesta fue satisfactoria
if response.status_code == 200:
    posts_data = response.json()
    print(f"Total de registros recuperados: {len(posts_data)}")
    print("\nEjemplo del primer registro (JSON puro):")
    print(json.dumps(posts_data[0], indent=2))
else:
    print(f"Error al consumir la API: {response.status_code}")

## 2. Conversión de Respuesta JSON a DataFrame de Pandas

In [ ]:
# Convertir la lista de objetos JSON en un DataFrame tabular
df_posts = pd.DataFrame(posts_data)
print(f"Dimensiones del DataFrame de posts: {df_posts.shape}")
df_posts.head()

In [ ]:
# Información técnica de los tipos de datos
df_posts.info()

## 3. Consumo de Estructuras JSON Anidadas con `pd.json_normalize` (`/users`)

En muchas APIs REST, los datos vienen con diccionarios y listas anidadas (por ejemplo: `address` que a su vez contiene `geo`). `pd.json_normalize()` aplana automáticamente estas jerarquías creando columnas compuestas (`address.street`, `address.geo.lat`, etc.).

In [ ]:
# Consumir el endpoint de usuarios (/users)
url_users = f"{BASE_URL}/users"
response_users = requests.get(url_users, timeout=10)
users_data = response_users.json()

# Aplanar estructuras jerárquicas anidadas
df_users = pd.json_normalize(users_data)
print(f"Dimensiones del DataFrame de usuarios: {df_users.shape}")
print("\nColumnas generadas por normalización:")
print(df_users.columns.tolist())

# Visualizar las columnas más relevantes
df_users[['id', 'name', 'username', 'email', 'address.city', 'company.name']].head()

## 4. Consumo de Tareas (`/todos`) y Cruce de Datos (Merge/Join)

In [ ]:
# Consumir el endpoint de tareas pendientes (/todos)
url_todos = f"{BASE_URL}/todos"
response_todos = requests.get(url_todos, timeout=10)
df_todos = pd.DataFrame(response_todos.json())

print(f"Total de tareas recuperadas: {len(df_todos)}")
df_todos.head()

In [ ]:
# Unir publicaciones (df_posts) con datos de autores (df_users)
df_posts_completos = pd.merge(
    df_posts,
    df_users[['id', 'name', 'email', 'address.city', 'company.name']],
    left_on='userId',
    right_on='id',
    how='left'
).drop(columns=['id_y']).rename(columns={'id_x': 'post_id'})

df_posts_completos.head()

## 5. Análisis y Métricas de los Datos Obtenidos

In [ ]:
# Total de posts publicados por autor
posts_por_usuario = df_posts_completos.groupby(['userId', 'name', 'company.name'])['post_id'].count().reset_index()
posts_por_usuario.columns = ['userId', 'nombre_autor', 'empresa', 'total_posts']
posts_por_usuario

In [ ]:
# Tasa de finalización de tareas por usuario
resumen_tareas = df_todos.groupby('userId')['completed'].agg(
    total_tareas='count',
    tareas_completadas='sum',
    porcentaje_cumplimiento='mean'
).reset_index()

resumen_tareas['porcentaje_cumplimiento'] = (resumen_tareas['porcentaje_cumplimiento'] * 100).round(1).astype(str) + '%'
resumen_tareas

## 6. Parámetros de Consulta (Query Params) y Manejo Robusto de Errores

In [ ]:
# Filtrar resultados directamente desde el servidor mediante query params
params = {'userId': 1}
response_filtrada = requests.get(f"{BASE_URL}/posts", params=params, timeout=10)
print(f"URL generada con query string: {response_filtrada.url}")
print(f"Publicaciones del usuario 1: {len(response_filtrada.json())}")

# Buenas prácticas: Gestión de excepciones HTTP con try-except
try:
    url_inexistente = f"{BASE_URL}/recurso_no_existe"
    resp_err = requests.get(url_inexistente, timeout=5)
    resp_err.raise_for_status()  # Provoca HTTPError si el código es 4xx o 5xx
except requests.exceptions.HTTPError as http_error:
    print(f"Excepción HTTP controlada: {http_error} (Código de estado: {resp_err.status_code})")
except requests.exceptions.RequestException as err:
    print(f"Fallo de conexión o red: {err}")

## 7. Exportación de los Datos Ingeridos desde la API

In [ ]:
# Exportar los conjuntos consolidados a formato CSV
df_posts_completos.to_csv('data/api_posts_usuarios.csv', index=False)
df_todos.to_csv('data/api_todos.csv', index=False)

print("Archivos exportados exitosamente en 'data/':")
print("  - data/api_posts_usuarios.csv")
print("  - data/api_todos.csv")